# Airfoil CL/CD Optimisation — full pipeline

Maximises **CL / CD** vs the kc135 baseline at fixed flow conditions (Re = 1e6, AoA = 10°), subject to:

- `CL ≥ 0.10`             (positive lift, with margin)
- thickness profile `≤ 0.9 × kc135` at x = 0.2, 0.4, 0.6, 0.8     (10 % thinner everywhere)
- geometric coherence       (`max_camber ≥ c25/c50/c75`, aft-monotone thinning)
- physical bounds           (LE radius > 0, thickness > 0, camber position ∈ [0, 1])

Runs **SLSQP**, **COBYQA**, and **trust-constr**, tracks every function evaluation and the maximum constraint violation at that point, then plots both histories and reconstructs the airfoil shapes.

All heavy lifting is in four helper modules in this folder:

| file                       | what's in it |
|----------------------------|--------------|
| `airfoil_opt_utils.py`     | model loader, kc135 baseline, bounds, scaler, problem builder |
| `tracked_optimisation.py`  | per-evaluation history wrapper + the three solver runners + plot helpers |
| `plot_airfoil.py`          | reconstructs upper/lower surfaces from the 12-feature vector |
| `evaluate_model.py`        | re-does the train/test split and reports R² / MAE / RMSE |

Run all cells top to bottom.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import warnings
import numpy as np
import matplotlib.pyplot as plt

from airfoil_opt_utils import (
    build_problem, KC135_GEOM, GEOM_NAMES, COHERENCE_LABELS,
)
from tracked_optimisation import (
    run_slsqp, run_cobyqa, run_trust_constr,
    plot_objective_history, plot_violation_history,
    summary_table, save_results,
)
from plot_airfoil    import plot_airfoils
from evaluate_model  import evaluate

%matplotlib inline
warnings.filterwarnings("ignore", message="delta_grad == 0")  # benign trust-constr chatter
np.set_printoptions(precision=4, suppress=True)
print("modules loaded")

## 0. Surrogate accuracy — R² on the train/test split

Replicates the original preprocessing (drop NaNs, log-transform CD), redoes the same `train_test_split(test_size=0.2, random_state=42)` that produced the saved model, and reports R² / MAE / RMSE for:

- the three trained outputs:  CL, log(CD), CM
- the de-normalised CD (after `exp(log CD)`)
- the **derived CL/CD** — this is what the optimiser actually maximises, so its R² is what you should trust the most

The parity plot is on the test set (n ≈ 569). A well-trained model puts every dot on the red diagonal.

In [ ]:
metrics = evaluate(plot=True, show=True, save_path="model_parity.png")

## 1. Build the problem

`build_problem` loads the trained `airfoil_surrogate.pt`, constructs the 12-D geometry-only design space, applies physical lower/upper bounds, and pre-builds the constraint lists (one for SLSQP, one for COBYQA / trust-constr). Tweak the kwargs below to change the problem.

In [ ]:
prob = build_problem(
    thickness_reduction = 0.10,   # 10 % thinner than kc135 at every sampled station
    cl_min              = 0.10,   # CL must stay ≥ this
    bounds_slack        = 0.02,   # widen training-data bounds by 2 % of span
    enable_coherence    = True,   # add the 5 linear coherence inequalities
)

cl_b, cd_b, cm_b = prob["kc135_pred"]
print(f"kc135 baseline   : CL={cl_b:.4f}  CD={cd_b:.6f}  CL/CD={cl_b/cd_b:.3f}  CM={cm_b:.4f}")
print(f"thickness target : {prob['thk_targets']}")
print(f"CL minimum       : {prob['cl_min']}")
print(f"design dim       : {len(prob['u0'])} (geometry-only, in u-space [0,1])")
print(f"coherence rules  :")
for c in COHERENCE_LABELS:
    print(f"  - {c}")

## 2. Run all three optimisers

Each `run_*` returns `(scipy_result, TrackingProblem)`. The tracker has logged the objective and the max-constraint-violation **at every function evaluation** scipy made — that's what the convergence plots in the next cells use.

In [ ]:
print("running SLSQP …");        res_s, tr_s = run_slsqp(prob)
print("running COBYQA …");       res_c, tr_c = run_cobyqa(prob)
print("running trust-constr …"); res_t, tr_t = run_trust_constr(prob)

results  = {"SLSQP": (res_s, tr_s),
            "COBYQA": (res_c, tr_c),
            "trust-constr": (res_t, tr_t)}
trackers = {name: tr for name, (_, tr) in results.items()}

for name, (res, tr) in results.items():
    print(f"  {name:<14s}  success={bool(res.success)}  "
          f"CL/CD={-res.fun:.3f}  evaluations={len(tr.f_calls)}")

## 3. Objective history — CL/CD at every function evaluation

x-axis = the k-th call scipy made to the objective. y-axis = CL/CD predicted by the surrogate at that point. Dashed line = kc135 baseline.

In [ ]:
_ = plot_objective_history(trackers, baseline_clcd=cl_b/cd_b,
                           save_path="objective_history.png")

## 4. Constraint violation history

For every function evaluation, the maximum violation over **all** inequality constraints (CL minimum, thickness profile, coherence, bounds) is recorded:

$$\text{violation}(u_k) \;=\; \max_i \, \max\bigl(0,\, -g_i(u_k)\bigr)$$

Plotted on a log scale. The dotted reference line is at 1e-6 — anything below counts as practically feasible.

In [ ]:
_ = plot_violation_history(trackers, save_path="violation_history.png")

## 5. Summary table

In [ ]:
summary_table(prob, results)

## 6. Save the optimised geometries

In [ ]:
saved = save_results(prob, results, out_dir=".")

## 7. Reconstruct and plot the airfoil shapes

Top panel: upper + lower surfaces (PCHIP through the 4 thickness samples and 3 camber samples; closed at LE and TE; cosine x-spacing). Open circles = thickness samples for the kc135 baseline; open squares = camber samples. Dashed = mean camber line.

Bottom panel: full thickness profile of every airfoil overlaid for direct comparison.

In [ ]:
geoms = {"kc135 baseline": KC135_GEOM, **saved}
_ = plot_airfoils(geoms, save_path="airfoil_shapes.png")

## 8. Per-solver detailed report

Calls `report_result` from `airfoil_opt_utils.py` for each solver — feature-by-feature delta vs kc135, plus per-constraint OK / VIOLATED status.

In [ ]:
from airfoil_opt_utils import report_result
for name, (res, tr) in results.items():
    report_result(name, res, prob)